# Economic Intelligence — modeling decisions (AI-02)

## Data pipeline
- **Marketplace:** `SNOWFLAKE_PUBLIC_DATA_FREE` — financial indicators + `COMPANY_RELATIONSHIPS`.
- **Curated views:** `V_UNEMPLOYMENT`, `V_RETAIL_SALES`, `V_INTEREST_RATES`, `V_INDUSTRIAL_PRODUCTION`, `V_CPI`, `V_GDP`, `V_COMPANY_RELATIONSHIPS`.
- **Cortex YAML:** logical tables map to `V_*` views plus `macro_wide` on `ECONOMIC_INDICATORS_WIDE` (values include `CPI_INDEX`, `GDP_VALUE` on `cpi` / `gdp`).
- **Macro panel:** `hackathon/sql/02_economic_indicators_wide.sql` → `ECONOMIC_INDICATORS_WIDE`; semantic logical table **`macro_wide`** for multi-indicator timelines. Optional extra Streamlit KPIs only if you wire them.

## Semantic YAML
- **File:** `hackathon/semantic_models/semantic_model.yaml` — multi-table + root `verified_queries`.
- **Stage:** `PUT` to `@HACKATHON.DATA.SEMANTIC_MODELS`

## Validation SQL (all curated views)

```sql
SELECT COUNT(*) FROM HACKATHON.DATA.V_UNEMPLOYMENT;
SELECT COUNT(*) FROM HACKATHON.DATA.V_RETAIL_SALES;
SELECT COUNT(*) FROM HACKATHON.DATA.V_INTEREST_RATES;
SELECT COUNT(*) FROM HACKATHON.DATA.V_INDUSTRIAL_PRODUCTION;
SELECT COUNT(*) FROM HACKATHON.DATA.V_CPI;
SELECT COUNT(*) FROM HACKATHON.DATA.V_GDP;
SELECT COUNT(*) FROM HACKATHON.DATA.V_COMPANY_RELATIONSHIPS;
SELECT COUNT(*) FROM HACKATHON.DATA.ECONOMIC_INDICATORS_WIDE;
```

## NL test log

Structured natural-language tests and expected outcomes: **`hackathon/QUERY_LOG.md`**. Update that file after you validate prompts in Snowflake.

## Five granular logical tables plus `macro_wide`

- **Source shape:** Snowflake Public Data exposes a long `financial_economic_indicators_timeseries` table keyed by `VARIABLE` / `VARIABLE_NAME` with heterogeneous measures (unemployment %, retail USD, rates, production index).
- **Curated views:** Each `V_*` applies tight `WHERE` filters on attributes (agency, measure, frequency, seasonal adjustment) so each Analyst logical table has a **consistent grain** (monthly macro or parent–subsidiary edge).
- **Why not only one wide table:** A single denormalized table of raw long-series IDs is hard to query in NL; we expose **five granular** logical tables for domain-specific filters (`VARIABLE_NAME`, `UNIT`, etc.) plus **`macro_wide`** on a **curated** pre-aggregated panel for judge-friendly joins.
- **Curated wide view:** `02_economic_indicators_wide.sql` aggregates each `V_*` to one row per `GEO_ID` + date so judges see a real **join story** (`macro_wide`) without fragile cross-joins of raw long tables. Granular logical tables remain primary for sector-specific NL.

## Joins (rubric alignment)

- **Within each macro view:** SQL may join `timeseries` to `attributes` in the underlying marketplace (already done inside each `V_*` definition). Analyst-generated SQL aggregates and filters **within** one logical table.
- **Across macro domains:** There is **no** shared entity key between unemployment, retail, rates, and industrial production in this design—only **calendar date** aligns. The YAML `custom_instructions` tell Analyst **not** to invent cross-table joins unless the user insists and a valid key exists.
- **Comparative questions** (e.g. unemployment *and* retail or *and* CPI): Use **`macro_wide`** / `ECONOMIC_INDICATORS_WIDE` when both measures exist on the panel. Otherwise prefer two questions, two queries, or a single-table proxy (e.g. retail MoM before/after hikes using only `V_RETAIL_SALES`).
- **Company graph:** Joins are along `COMPANY_RELATIONSHIPS` edges; the view `V_COMPANY_RELATIONSHIPS` is restricted to `RELATIONSHIP_TYPE = 'PARENT'` for clear parent → subsidiary direction.

## Iteration after failures (YAML first, then app)

1. **Symptom:** Analyst returns wrong SQL, refuses, or invalid CTEs (especially company counts).
2. **First lever:** Extend `custom_instructions`, synonyms, and **`verified_queries`** in `semantic_model.yaml`; re-`PUT` to `@HACKATHON.DATA.SEMANTIC_MODELS`.
3. **Second lever:** Add a narrow intent in `streamlit_economic_intelligence/app.py` → `_fallback_sql_for_question()` only for **high-value** patterns that still fail after YAML updates.
4. **Track:** Record pass/partial/fail in `hackathon/QUERY_LOG.md` so judges and teammates see what was validated.

**Ambiguous asks** (“Is the economy doing well?”, “Show me rates”): Expect **clarifying** behavior or a default series; improve routing with synonyms rather than forcing one brittle SQL shape.

## Reproduce: views → stage → app

1. Run `hackathon/economic_indicators_views.sql` then `hackathon/sql/02_economic_indicators_wide.sql` (required for `macro_wide` / compare-over-time verified queries).
2. Run `hackathon/sql/03_semantic_stage.sql` to ensure stage `HACKATHON.DATA.SEMANTIC_MODELS` exists.
3. Upload the model (from a local clone path):

   ```sql
   PUT file://semantic_model.yaml @HACKATHON.DATA.SEMANTIC_MODELS
     AUTO_COMPRESS=FALSE OVERWRITE=TRUE;
   ```

4. Deploy Streamlit files per `streamlit_economic_intelligence/README.md` (§8 Deployment); point Analyst at the stage path your app uses.
5. Execute validation SQL in the first cell and spot-check `hackathon/QUERY_LOG.md` prompts in the live app.